# 04 Primordial Flexible-Centrality 8 TeV (Do-Not-Touch Legacy)

This notebook is an **upgrade layer** on top of current working notebooks.

- Does **not modify** `00_primordial_single_set_8TeV.ipynb`, `01_...executed.ipynb`, `03_primordial_compare_8TeV_all_forms.ipynb`
- Uses upgraded primordial + eloss-Glauber bridge
- Supports flexible centrality bin edges with physically motivated weighting
- Produces publication-ready primordial-only plots and saves them

## Physics defaults used here
- Centrality aggregation weight: `Ncoll` (`maps.b_to_nbin`) per underlying `b` sample
- MB computed from centrality-slice weighted average (same weight scheme)

You can change these switches in the config cell.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

project = Path.cwd()
if not (project / 'primordial_code').exists():
    project = project.parent

sys.path.insert(0, str(project))
sys.path.insert(0, str(project / 'primordial_code'))
sys.path.insert(0, str(project / 'primordial_notebooks'))

import primordial_only_eloss_glauber_test as prim
from primordial_module import ReaderConfig, Style, make_bins_from_width


In [ ]:
# ================================
# User configuration (8 TeV only)
# ================================
ENERGY = '8.16'
FORMS = ('new', 'old', 'radius')
MODELS = ('Pert', 'NPWLC')

# Flexible centrality edges (percent)
CENTRALITY_EDGE_SETS = {
    'fine_5to100': [0, 5, 10, 20, 40, 60, 80, 100],
    'std_10to100': [0, 10, 20, 40, 60, 80, 100],
    'coarse_20to100': [0, 20, 40, 60, 80, 100],
}
CENT_KEY = 'fine_5to100'
CENT_EDGES = CENTRALITY_EDGE_SETS[CENT_KEY]

# Weighting scheme in centrality aggregation
# - 'nbin' : Ncoll-weighted (recommended, consistent with current notebooks)
# - 'flat' : equal b-point weight
CENT_WEIGHT_SCHEME = 'nbin'

# Plot + save
SAVE_PDF = True
SAVE_CSV = True
DPI = 150
ALPHA_BAND = 0.22
Y_LIM = (0.0, 1.0)   # default requested; override if needed

# Binning and windows
Y_BINS = make_bins_from_width(-5.0, 5.0, 0.5)
PT_BINS = make_bins_from_width(0.0, 20.0, 2.5)
PT_WINDOW = (0.0, 15.0)
Y_WINDOWS = prim.Y_WINDOWS

# States
STATES = ['jpsi_1S', 'chicJ_1P', 'psi_2S']
STATE_GROUPS = [
    ('jpsi_1S', 'psi_2S'),
    ('jpsi_1S', 'chicJ_1P', 'psi_2S'),
]

# Color conventions requested
prim.MODEL_COLORS['Pert'] = 'tab:blue'
prim.MODEL_COLORS['NPWLC'] = 'tab:green'
STATE_COLORS = {
    'jpsi_1S': 'tab:blue',
    'chicJ_1P': 'tab:orange',
    'psi_2S': 'tab:purple',
}

# Style
Style.apply()
mpl.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'legend.frameon': True,
    'legend.framealpha': 0.85,
    'axes.unicode_minus': False,
})

outdir = project / 'primordial_output' / 'flexible_8TeV'
outdir.mkdir(exist_ok=True, parents=True)

assert sorted(CENT_EDGES) == list(CENT_EDGES), 'CENT_EDGES must be sorted ascending'
assert CENT_EDGES[0] == 0 and CENT_EDGES[-1] == 100, 'CENT_EDGES must start at 0 and end at 100'
assert len(set(CENT_EDGES)) == len(CENT_EDGES), 'CENT_EDGES must not repeat values'

print('Using CENT_EDGES:', CENT_EDGES)
print('Weight scheme:', CENT_WEIGHT_SCHEME)


In [ ]:
# Build centrality class tuples from edges
CENT_CLASSES = list(zip(CENT_EDGES[:-1], CENT_EDGES[1:]))
print('CENT_CLASSES:', CENT_CLASSES)


In [ ]:
# Step 1: Generate/validate Glauber maps from eloss optical Glauber
sqrts_gev = 8160.0
glauber_root = project / 'primordial_output' / 'glauber_data' / '8TeV_eloss'
prim.generate_primordial_glauber_maps(
    glauber_root,
    cfg=prim.GlauberBridgeConfig(
        roots_gev=sqrts_gev,
        target_a=208,
        system='pA',
        bmax_fm=20.0,
        nb=401,
        include_npart=True,
        verbose=False,
    ),
)
prim._validate_glauber_maps(glauber_root)
print('[ok] glauber maps:', glauber_root)


In [ ]:
# Step 2: Load all forms (each form loads lower/upper run pair) and validate mappings
cfg = ReaderConfig(debug=False)
combos_by_form = {}

for form in FORMS:
    combos = prim._load_primordial_combos(ENERGY, form, glauber_root, cfg)
    combos = [c for c in combos if c.model in MODELS]
    assert combos, f'No combos for form={form}'
    combos_by_form[form] = combos

    print(f'\n[form={form}] loaded:', ', '.join([f'{c.model}:{c.form}' for c in combos]))
    for c in combos:
        maps = prim._maps_from_runs(c.runs)
        probe = maps.b_to_nbin(np.array([0.0, 2.0, 4.0], float))
        assert np.all(np.isfinite(probe)), f'Non-finite nbin probe for {form}/{c.model}: {probe}'
        print(f'  [ok] {c.model} nbin(b=0,2,4) = {probe[0]:.3f}, {probe[1]:.3f}, {probe[2]:.3f}')


In [ ]:
# Step 3: Compute primordial outputs for each form using flexible centrality classes

def primordial_vs_y_all_weighted(combos, states=STATES, cent_classes=CENT_CLASSES, pt_window=PT_WINDOW, y_bins=Y_BINS, weight_scheme='nbin'):
    results = {c.model: {} for c in combos}
    for combo in combos:
        maps = prim._maps_from_runs(combo.runs)
        dfC_b, dfB_b = combo.ens.central_and_band_vs_y_per_b(
            pt_window=pt_window,
            y_bins=y_bins,
            with_feeddown=True,
            use_nbin=True,
            flip_y=True,
        )
        assert not dfC_b.empty, f'Empty y-per-b table for {combo.model}'

        slice_w = prim._centrality_slice_weights(maps, dfC_b, cent_classes, scheme=weight_scheme)
        cent_tags_for_weights = []

        for ic, (lo, hi) in enumerate(cent_classes):
            tag = f"{int(lo)}–{int(hi)}%"
            dfc, dfb = prim._aggregate_class(dfC_b, dfB_b, maps, lo, hi, states, 'y', weight=weight_scheme)
            if dfc is None:
                continue

            y_cent = dfc['y'].to_numpy(float)
            entry = {}
            for s in states:
                Rc = dfc[s].to_numpy(float)
                if dfb is not None and f'{s}_lo' in dfb:
                    Rlo = dfb[f'{s}_lo'].to_numpy(float)
                    Rhi = dfb[f'{s}_hi'].to_numpy(float)
                else:
                    Rlo = Rc
                    Rhi = Rc
                entry[s] = (Rc, Rlo, Rhi, y_cent)

            results[combo.model][tag] = entry
            cent_tags_for_weights.append((tag, ic))

        # MB from centrality slices
        tag_mb = '0–100% (MB)'
        entry_mb = {}
        for s in states:
            Rc_slices, Rlo_slices, Rhi_slices = [], [], []
            y_slices, w_used = [], []

            for tag, ic in cent_tags_for_weights:
                if s not in results[combo.model][tag]:
                    continue
                Rc, Rlo, Rhi, y_cent = results[combo.model][tag][s]
                Rc_slices.append(Rc)
                Rlo_slices.append(Rlo)
                Rhi_slices.append(Rhi)
                y_slices.append(y_cent)
                w_used.append(slice_w[ic])

            if not Rc_slices:
                continue

            W = np.asarray(w_used, float)
            W /= W.sum()

            # align on common y grid
            y_round_lists = [np.round(y_arr, 6) for y_arr in y_slices]
            common_vals = set(y_round_lists[0])
            for yr in y_round_lists[1:]:
                common_vals &= set(yr)
            if not common_vals:
                continue

            y_common = np.array(sorted(common_vals))
            Rc_stack, Rlo_stack, Rhi_stack = [], [], []
            for Rc, Rlo, Rhi, y_cent in zip(Rc_slices, Rlo_slices, Rhi_slices, y_slices):
                y_round = np.round(y_cent, 6)
                idxs = [np.where(y_round == v)[0][0] for v in y_common]
                Rc_stack.append(Rc[idxs])
                Rlo_stack.append(Rlo[idxs])
                Rhi_stack.append(Rhi[idxs])

            Rc_stack = np.stack(Rc_stack, axis=0)
            Rlo_stack = np.stack(Rlo_stack, axis=0)
            Rhi_stack = np.stack(Rhi_stack, axis=0)

            Rc_MB = np.tensordot(W, Rc_stack, axes=(0, 0))
            Rlo_MB = np.tensordot(W, Rlo_stack, axes=(0, 0))
            Rhi_MB = np.tensordot(W, Rhi_stack, axes=(0, 0))
            entry_mb[s] = (Rc_MB, Rlo_MB, Rhi_MB, y_common)

        if entry_mb:
            results[combo.model][tag_mb] = entry_mb

    return results


def primordial_vs_pT_all_weighted(combos, states=STATES, cent_classes=CENT_CLASSES, y_wins=Y_WINDOWS, pt_bins=PT_BINS, weight_scheme='nbin'):
    out = {}
    for yname, y0, y1 in prim._iter_ywins(y_wins):
        out[yname] = {c.model: {} for c in combos}
        for combo in combos:
            maps = prim._maps_from_runs(combo.runs)
            dfC_b, dfB_b = combo.ens.central_and_band_vs_pt_per_b(
                y_window=(y0, y1, yname),
                pt_bins=pt_bins,
                with_feeddown=True,
                use_nbin=True,
            )
            assert not dfC_b.empty, f'Empty pT-per-b table for {combo.model}/{yname}'

            slice_w = prim._centrality_slice_weights(maps, dfC_b, cent_classes, scheme=weight_scheme)
            cent_tags_for_weights = []

            for ic, (lo, hi) in enumerate(cent_classes):
                tag = f"{int(lo)}–{int(hi)}%"
                dfc, dfb = prim._aggregate_class(dfC_b, dfB_b, maps, lo, hi, states, 'pt', weight=weight_scheme)
                if dfc is None:
                    continue

                pT_cent = dfc['pt'].to_numpy(float)
                entry = {}
                for s in states:
                    Rc = dfc[s].to_numpy(float)
                    if dfb is not None and f'{s}_lo' in dfb:
                        Rlo = dfb[f'{s}_lo'].to_numpy(float)
                        Rhi = dfb[f'{s}_hi'].to_numpy(float)
                    else:
                        Rlo = Rc
                        Rhi = Rc
                    entry[s] = (Rc, Rlo, Rhi, pT_cent)
                out[yname][combo.model][tag] = entry
                cent_tags_for_weights.append((tag, ic))

            # MB from centrality slices
            tag_mb = '0–100% (MB)'
            entry_mb = {}
            for s in states:
                pT_ref = None
                Rc_slices, Rlo_slices, Rhi_slices = [], [], []
                w_used = []
                for tag, ic in cent_tags_for_weights:
                    if s not in out[yname][combo.model].get(tag, {}):
                        continue
                    Rc, Rlo, Rhi, pT_cent = out[yname][combo.model][tag][s]
                    if pT_ref is None:
                        pT_ref = pT_cent
                    Rc_slices.append(Rc)
                    Rlo_slices.append(Rlo)
                    Rhi_slices.append(Rhi)
                    w_used.append(slice_w[ic])

                if not Rc_slices:
                    continue

                W = np.asarray(w_used, float)
                W /= W.sum()
                min_len = min(len(arr) for arr in Rc_slices)
                pT_ref_common = pT_ref[:min_len]

                Rc_stack = np.stack([arr[:min_len] for arr in Rc_slices], axis=0)
                Rlo_stack = np.stack([arr[:min_len] for arr in Rlo_slices], axis=0)
                Rhi_stack = np.stack([arr[:min_len] for arr in Rhi_slices], axis=0)

                Rc_MB = np.tensordot(W, Rc_stack, axes=(0, 0))
                Rlo_MB = np.tensordot(W, Rlo_stack, axes=(0, 0))
                Rhi_MB = np.tensordot(W, Rhi_stack, axes=(0, 0))
                entry_mb[s] = (Rc_MB, Rlo_MB, Rhi_MB, pT_ref_common)

            if entry_mb:
                out[yname][combo.model][tag_mb] = entry_mb

    return out


results_by_form = {}
for form, combos in combos_by_form.items():
    print(f'Computing form={form} ...')
    prim_y = primordial_vs_y_all_weighted(combos, weight_scheme=CENT_WEIGHT_SCHEME)
    prim_pt = primordial_vs_pT_all_weighted(combos, weight_scheme=CENT_WEIGHT_SCHEME)
    prim_cent = prim.primordial_vs_cent_from_vs_y(prim_y, Y_WINDOWS, STATES, MODELS)
    results_by_form[form] = {
        'combos': combos,
        'prim_y': prim_y,
        'prim_pt': prim_pt,
        'prim_cent': prim_cent,
    }
    print(f'[ok] {form}: y/pT/cent computed')

print('Done forms:', list(results_by_form.keys()))


In [ ]:
# Step 4: plotting helpers (publication-style, no titles, shared axes, step bands)

def add_info_box(ax, state, energy, form, y_window_text=None):
    tau = prim._tauform_for_form(form)
    tau_lo, tau_hi = tau[state]
    lines = [
        rf"{prim.STATE_LABELS[state]}, $\sqrt{{s_{{NN}}}}={float(energy):.2f}$ TeV",
        rf"$\tau_{{\rm form}}={tau_lo:.2g}\text{{–}}{tau_hi:.2g}$ fm",
    ]
    if y_window_text is not None:
        lines.append(y_window_text)
    txt = "\n".join(lines)
    ax.text(
        0.02, 0.96, txt,
        transform=ax.transAxes,
        ha='left', va='top', fontsize=10,
        bbox=dict(facecolor='white', alpha=0.78, edgecolor='none')
    )


def plot_vs_y_panels(prim_y_form, state, form, energy=ENERGY, save=True):
    cent_tags = prim._sorted_cent_tags(next(iter(prim_y_form.values())))
    n = len(cent_tags)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.9*ncols, 3.0*nrows),
        dpi=DPI,
        sharex=True,
        sharey=True,
        gridspec_kw={'wspace': 0.02, 'hspace': 0.05}
    )
    axes = np.atleast_1d(axes).ravel()

    for i, tag in enumerate(cent_tags):
        ax = axes[i]
        for model in MODELS:
            if model not in prim_y_form or tag not in prim_y_form[model] or state not in prim_y_form[model][tag]:
                continue
            Rc, Rlo, Rhi, y_cent = prim_y_form[model][tag][state]
            x_edges, y_c = prim.step_from_centers(y_cent, Rc)
            _, y_lo = prim.step_from_centers(y_cent, Rlo)
            _, y_hi = prim.step_from_centers(y_cent, Rhi)
            lab = model if i == 0 else None
            col = prim.MODEL_COLORS[model]
            ls = prim.MODEL_LS[model]
            ax.step(x_edges, y_c, where='post', color=col, ls=ls, lw=1.8, label=lab)
            ax.fill_between(x_edges, y_lo, y_hi, step='post', color=col, alpha=ALPHA_BAND, linewidth=0.0)

        ax.set_xlim(-5, 5)
        ax.set_ylim(*Y_LIM)
        ax.grid(False)
        ax.minorticks_on()
        ax.tick_params(axis='y', which='major', direction='in', right=True)
        ax.tick_params(axis='y', which='minor', direction='in', right=True)
        ax.text(0.02, 0.96, tag, transform=ax.transAxes, ha='left', va='top', fontsize=10,
                bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))

        if i % ncols == 0:
            ax.set_ylabel(r'$R_{pA}$')
        if i >= (nrows-1)*ncols:
            ax.set_xlabel(r'$y$')

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    h, l = axes[0].get_legend_handles_labels()
    if h:
        axes[0].legend(h, l, loc='lower right', frameon=False, fontsize=9)
    add_info_box(axes[0], state, energy, form)

    fig.tight_layout()
    if save:
        tag = f"{energy.replace('.', 'p')}TeV_{form}_{CENT_KEY}"
        fig.savefig(outdir / f"primordial_flex_RpA_vs_y_{state}_{tag}.pdf", bbox_inches='tight')
    plt.show()
    plt.close(fig)


def plot_vs_pt_panels(prim_pt_form, state, form, yname, y0, y1, energy=ENERGY, save=True):
    models_block = prim_pt_form[yname]
    cent_tags = prim._sorted_cent_tags(next(iter(models_block.values())))
    n = len(cent_tags)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.9*ncols, 3.0*nrows),
        dpi=DPI,
        sharex=True,
        sharey=True,
        gridspec_kw={'wspace': 0.02, 'hspace': 0.05}
    )
    axes = np.atleast_1d(axes).ravel()

    for i, tag in enumerate(cent_tags):
        ax = axes[i]
        for model in MODELS:
            if model not in models_block or tag not in models_block[model] or state not in models_block[model][tag]:
                continue
            Rc, Rlo, Rhi, pT_cent = models_block[model][tag][state]
            x_edges, y_c = prim.step_from_centers(pT_cent, Rc)
            _, y_lo = prim.step_from_centers(pT_cent, Rlo)
            _, y_hi = prim.step_from_centers(pT_cent, Rhi)
            lab = model if i == 0 else None
            col = prim.MODEL_COLORS[model]
            ls = prim.MODEL_LS[model]
            ax.step(x_edges, y_c, where='post', color=col, ls=ls, lw=1.8, label=lab)
            ax.fill_between(x_edges, y_lo, y_hi, step='post', color=col, alpha=ALPHA_BAND, linewidth=0.0)

        ax.set_xlim(*PT_WINDOW)
        ax.set_ylim(*Y_LIM)
        ax.grid(False)
        ax.minorticks_on()
        ax.tick_params(axis='y', which='major', direction='in', right=True)
        ax.tick_params(axis='y', which='minor', direction='in', right=True)
        ax.text(0.02, 0.96, tag, transform=ax.transAxes, ha='left', va='top', fontsize=10,
                bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))

        if i % ncols == 0:
            ax.set_ylabel(r'$R_{pA}$')
        if i >= (nrows-1)*ncols:
            ax.set_xlabel(r'$p_T$ [GeV]')

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    h, l = axes[0].get_legend_handles_labels()
    if h:
        axes[0].legend(h, l, loc='lower right', frameon=False, fontsize=9)

    add_info_box(axes[0], state, energy, form, y_window_text=rf"${y0:.2f} < y < {y1:.2f}$")

    fig.tight_layout()
    if save:
        tag = f"{energy.replace('.', 'p')}TeV_{form}_{CENT_KEY}"
        fig.savefig(outdir / f"primordial_flex_RpA_vs_pT_{state}_{yname}_{tag}.pdf", bbox_inches='tight')
    plt.show()
    plt.close(fig)


def plot_vs_cent(prim_cent_form, state, form, energy=ENERGY, save=True):
    ywins = list(prim._iter_ywins(Y_WINDOWS))
    fig, axes = plt.subplots(
        1, len(ywins), figsize=(5.0*len(ywins), 3.6), dpi=DPI,
        sharex=True, sharey=True, gridspec_kw={'wspace': 0.03}
    )
    axes = np.atleast_1d(axes)
    edges = np.array(CENT_EDGES, float)

    for iax, (yname, y0, y1) in enumerate(ywins):
        ax = axes[iax]
        if yname not in prim_cent_form:
            ax.set_visible(False)
            continue

        for model in MODELS:
            if model not in prim_cent_form[yname] or state not in prim_cent_form[yname][model]:
                continue
            cent_mid, Rc, Rlo, Rhi, Rc_MB, Rlo_MB, Rhi_MB = prim_cent_form[yname][model][state]

            y_c = np.concatenate([Rc, Rc[-1:]])
            y_lo = np.concatenate([Rlo, Rlo[-1:]])
            y_hi = np.concatenate([Rhi, Rhi[-1:]])
            col = prim.MODEL_COLORS[model]
            ls = prim.MODEL_LS[model]
            lab = model if iax == 0 else None

            ax.step(edges, y_c, where='post', color=col, ls=ls, lw=1.8, label=lab)
            ax.fill_between(edges, y_lo, y_hi, step='post', color=col, alpha=ALPHA_BAND, linewidth=0.0)

            # MB horizontal step-band + center line
            if np.isfinite(Rc_MB):
                x_mb = np.array([0.0, 100.0])
                ax.fill_between(x_mb, [Rlo_MB, Rlo_MB], [Rhi_MB, Rhi_MB],
                                step='post', color='none', hatch='////', edgecolor=col, linewidth=0.0)
                ax.step(x_mb, [Rc_MB, Rc_MB], where='post', color=col, ls='--', lw=1.5)

        ax.set_xlim(0.0, 100.0)
        ax.set_ylim(*Y_LIM)
        ax.set_xlabel('Centrality [%]')
        if iax == 0:
            ax.set_ylabel(r'$R_{pA}$')
        ax.grid(False)
        ax.minorticks_on()
        ax.tick_params(axis='y', which='major', direction='in', right=True)
        ax.tick_params(axis='y', which='minor', direction='in', right=True)
        ax.text(0.02, 0.96, rf"${y0:.2f} < y < {y1:.2f}$", transform=ax.transAxes,
                ha='left', va='top', fontsize=10,
                bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))

    h, l = axes[0].get_legend_handles_labels()
    if h:
        axes[0].legend(h, l, loc='lower right', frameon=False, fontsize=9)
    add_info_box(axes[0], state, energy, form)

    fig.tight_layout()
    if save:
        tag = f"{energy.replace('.', 'p')}TeV_{form}_{CENT_KEY}"
        fig.savefig(outdir / f"primordial_flex_RpA_vs_cent_{state}_{tag}.pdf", bbox_inches='tight')
    plt.show()
    plt.close(fig)


def export_tidy_csvs(prim_y_form, prim_pt_form, prim_cent_form, form, energy=ENERGY):
    tag = f"{energy.replace('.', 'p')}TeV_{form}_{CENT_KEY}"

    rows_y = []
    for model in MODELS:
        if model not in prim_y_form:
            continue
        for cent_tag in prim_y_form[model]:
            for state in STATES:
                if state not in prim_y_form[model][cent_tag]:
                    continue
                Rc, Rlo, Rhi, y_cent = prim_y_form[model][cent_tag][state]
                for yv, rc, lo, hi in zip(y_cent, Rc, Rlo, Rhi):
                    rows_y.append({
                        'energy': energy, 'form': form, 'model': model, 'centrality': cent_tag,
                        'state': state, 'y': yv, 'R': rc, 'R_lo': lo, 'R_hi': hi
                    })
    if rows_y:
        pd.DataFrame(rows_y).to_csv(outdir / f"primordial_flex_RpA_vs_y_{tag}.csv", index=False)

    rows_pt = []
    for yname in prim_pt_form:
        block = prim_pt_form[yname]
        for model in MODELS:
            if model not in block:
                continue
            for cent_tag in block[model]:
                for state in STATES:
                    if state not in block[model][cent_tag]:
                        continue
                    Rc, Rlo, Rhi, pT_cent = block[model][cent_tag][state]
                    for pv, rc, lo, hi in zip(pT_cent, Rc, Rlo, Rhi):
                        rows_pt.append({
                            'energy': energy, 'form': form, 'y_window': yname, 'model': model,
                            'centrality': cent_tag, 'state': state, 'pT': pv, 'R': rc, 'R_lo': lo, 'R_hi': hi
                        })
    if rows_pt:
        pd.DataFrame(rows_pt).to_csv(outdir / f"primordial_flex_RpA_vs_pT_{tag}.csv", index=False)

    rows_cent = []
    for yname in prim_cent_form:
        for model in MODELS:
            if model not in prim_cent_form[yname]:
                continue
            for state in STATES:
                if state not in prim_cent_form[yname][model]:
                    continue
                cent_mid, Rc, Rlo, Rhi, Rc_MB, Rlo_MB, Rhi_MB = prim_cent_form[yname][model][state]
                for c, rc, lo, hi in zip(cent_mid, Rc, Rlo, Rhi):
                    rows_cent.append({
                        'energy': energy, 'form': form, 'y_window': yname, 'model': model,
                        'state': state, 'cent_mid': c, 'R': rc, 'R_lo': lo, 'R_hi': hi,
                        'R_MB': Rc_MB, 'R_MB_lo': Rlo_MB, 'R_MB_hi': Rhi_MB
                    })
    if rows_cent:
        pd.DataFrame(rows_cent).to_csv(outdir / f"primordial_flex_RpA_vs_cent_{tag}.csv", index=False)


In [ ]:
# Step 5: Generate all requested plots + save
for form in FORMS:
    print(f'\n=== Plotting form={form} ===')
    prim_y_form = results_by_form[form]['prim_y']
    prim_pt_form = results_by_form[form]['prim_pt']
    prim_cent_form = results_by_form[form]['prim_cent']

    # State-wise full outputs
    for state in STATES:
        plot_vs_y_panels(prim_y_form, state=state, form=form, save=SAVE_PDF)
        for yname, y0, y1 in prim._iter_ywins(Y_WINDOWS):
            plot_vs_pt_panels(prim_pt_form, state=state, form=form, yname=yname, y0=y0, y1=y1, save=SAVE_PDF)
        plot_vs_cent(prim_cent_form, state=state, form=form, save=SAVE_PDF)

    # Export tidy tables
    if SAVE_CSV:
        export_tidy_csvs(prim_y_form, prim_pt_form, prim_cent_form, form=form)

print('\n[done] all plots generated and saved to:', outdir)


In [ ]:
# Step 6: Optional MB state-comparison plots (2-state and 3-state groups)

def get_mb_tag(model_block):
    tags = [t for t in model_block if 'MB' in t]
    return tags[0] if tags else None

for form in FORMS:
    prim_y_form = results_by_form[form]['prim_y']
    prim_pt_form = results_by_form[form]['prim_pt']
    print(f'\n=== MB state-compare form={form} ===')

    for model in MODELS:
        if model not in prim_y_form:
            continue
        mb_tag = get_mb_tag(prim_y_form[model])
        if mb_tag is None:
            continue

        for group in STATE_GROUPS:
            # MB vs y
            fig, ax = plt.subplots(figsize=(6.6, 4.3), dpi=DPI)
            for s in group:
                if s not in prim_y_form[model][mb_tag]:
                    continue
                Rc, Rlo, Rhi, y_cent = prim_y_form[model][mb_tag][s]
                x_edges, y_c = prim.step_from_centers(y_cent, Rc)
                _, y_lo = prim.step_from_centers(y_cent, Rlo)
                _, y_hi = prim.step_from_centers(y_cent, Rhi)
                col = STATE_COLORS[s]
                ax.step(x_edges, y_c, where='post', color=col, lw=1.8, label=prim.STATE_LABELS[s])
                ax.fill_between(x_edges, y_lo, y_hi, step='post', color=col, alpha=ALPHA_BAND, linewidth=0.0)

            ax.set_xlim(-5, 5)
            ax.set_ylim(*Y_LIM)
            ax.set_xlabel(r'$y$')
            ax.set_ylabel(r'$R_{pA}$')
            ax.grid(False)
            ax.minorticks_on()
            ax.tick_params(axis='y', which='major', direction='in', right=True)
            ax.tick_params(axis='y', which='minor', direction='in', right=True)
            ax.legend(frameon=False, ncol=1)
            ax.text(0.02, 0.96, rf"{model}, $\sqrt{{s_{{NN}}}}={float(ENERGY):.2f}$ TeV, MB", transform=ax.transAxes,
                    ha='left', va='top', fontsize=10, bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))

            if SAVE_PDF:
                gtag = '_'.join([s.replace('_', '') for s in group])
                tag = f"{ENERGY.replace('.', 'p')}TeV_{form}_{CENT_KEY}"
                fig.savefig(outdir / f"primordial_flex_MB_vs_y_statecmp_{model}_{gtag}_{tag}.pdf", bbox_inches='tight')
            plt.show()
            plt.close(fig)

            # MB vs pT in three rapidity windows
            for yname, y0, y1 in prim._iter_ywins(Y_WINDOWS):
                if yname not in prim_pt_form or model not in prim_pt_form[yname]:
                    continue
                mb_tag_pt = get_mb_tag(prim_pt_form[yname][model])
                if mb_tag_pt is None:
                    continue

                fig, ax = plt.subplots(figsize=(6.6, 4.3), dpi=DPI)
                for s in group:
                    if s not in prim_pt_form[yname][model][mb_tag_pt]:
                        continue
                    Rc, Rlo, Rhi, pT_cent = prim_pt_form[yname][model][mb_tag_pt][s]
                    x_edges, y_c = prim.step_from_centers(pT_cent, Rc)
                    _, y_lo = prim.step_from_centers(pT_cent, Rlo)
                    _, y_hi = prim.step_from_centers(pT_cent, Rhi)
                    col = STATE_COLORS[s]
                    ax.step(x_edges, y_c, where='post', color=col, lw=1.8, label=prim.STATE_LABELS[s])
                    ax.fill_between(x_edges, y_lo, y_hi, step='post', color=col, alpha=ALPHA_BAND, linewidth=0.0)

                ax.set_xlim(*PT_WINDOW)
                ax.set_ylim(*Y_LIM)
                ax.set_xlabel(r'$p_T$ [GeV]')
                ax.set_ylabel(r'$R_{pA}$')
                ax.grid(False)
                ax.minorticks_on()
                ax.tick_params(axis='y', which='major', direction='in', right=True)
                ax.tick_params(axis='y', which='minor', direction='in', right=True)
                ax.legend(frameon=False, ncol=1)
                ax.text(0.02, 0.96, rf"{model}, $\sqrt{{s_{{NN}}}}={float(ENERGY):.2f}$ TeV, MB", transform=ax.transAxes,
                        ha='left', va='top', fontsize=10, bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))
                ax.text(0.02, 0.88, rf"${y0:.2f} < y < {y1:.2f}$", transform=ax.transAxes,
                        ha='left', va='top', fontsize=10, bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))

                if SAVE_PDF:
                    gtag = '_'.join([s.replace('_', '') for s in group])
                    tag = f"{ENERGY.replace('.', 'p')}TeV_{form}_{CENT_KEY}"
                    fig.savefig(outdir / f"primordial_flex_MB_vs_pT_statecmp_{model}_{yname}_{gtag}_{tag}.pdf", bbox_inches='tight')
                plt.show()
                plt.close(fig)

print('\n[done] MB state-compare plots generated')


In [ ]:
# Step 7: quick artifact inventory
pdfs = sorted(outdir.glob('*.pdf'))
csvs = sorted(outdir.glob('*.csv'))
print('PDF count:', len(pdfs))
print('CSV count:', len(csvs))
print('Sample PDFs:')
for p in pdfs[:12]:
    print(' -', p.name)
print('Sample CSVs:')
for p in csvs[:12]:
    print(' -', p.name)
